# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content item for one client on one report date, identified by `report_date × client_hash_id × content_hash_id`.

**Working time window:** I use March 2026 as the development/verification month. The March partition contains 9,841,378 rows spanning 2026-03-01 through 2026-03-31. June 2026 is treated as a sealed final month rather than used for developing the contract.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

Candidate features are historical or currently available performance signals that can be known at the decision moment, such as:

* `gsc_impressions`
* `gsc_clicks`
* `gsc_sum_position`
* `sessions_ai`
* `scroll_events`

These are treated as candidate features only when their underlying data is available at the decision moment.

### Label / target proxy

The ranking target is whether a content item should be prioritized for attention based on its subsequent performance. A future-period performance change can be used as the target/proxy, but any field derived from that future outcome must not be used as an input feature.

### Context

The following fields provide context or identification rather than predictive signal:

* `report_date`
* `client_hash_id`
* `content_hash_id`
* `gsc_data_available`
* `ga4_data_available`

The client and content IDs are pseudonymous identifiers and are used for grouping, joining, and defining the observation grain.

### Deliberate exclusion

I deliberately exclude future-derived performance fields such as `trend_pct` or `trend_direction` from the feature set because they contain information about the outcome being predicted and would create target leakage.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



The following checks use March 2026 as the development/verification slice.

### Query 1 — Verify the grain

The expected grain is `report_date × client_hash_id × content_hash_id`. This query checks whether any combination occurs more than once.

```python
grain_query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_result = con.execute(grain_query).fetchdf()
display(grain_result)
```

**Observed result:** No duplicate combinations were returned, supporting the stated daily client × content grain.

### Query 2 — Verify the March slice size and date window

```python
march_query = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""

march_result = con.execute(march_query).fetchdf()
display(march_result)
```

**Observed result:** 9,841,378 rows, covering 2026-03-01 through 2026-03-31.

### Query 3 — Verify data availability

```python
availability_query = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""

availability_result = con.execute(availability_query).fetchdf()
display(availability_result)
```

**Observed result:** 9,841,378 total rows; 9,841,378 rows have GSC availability and 3,611,061 rows have GA4 availability.

The availability check uses `IS TRUE` so that availability flags are interpreted explicitly rather than treating missing or non-true values as available.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



A key limitation is that data availability and history depth can differ across clients. The availability flags show that not every March 2026 row has the same underlying source coverage: while GSC data is available across the March slice, GA4 data is available for only 3,611,061 of 9,841,378 rows.

This means features based on GA4 metrics should only be used when the corresponding availability flag indicates that the data is present. Missing source coverage should not automatically be interpreted as zero performance.

Another limitation is that the working month is only a development/verification slice. The final month should remain sealed so that feature and label decisions are not influenced by future test-period information.


## Self-check

✅ Every section above is filled — markdown thinking and supporting code are included
✅ No client names, URLs, or private queries — our content uses anonymized IDs and warehouse fields only.
